In [2]:
import numpy as np
import pandas as pd
import scipy

from IPython.display import clear_output

import sys
sys.path.append('../../../../Documents/GitHub/gustav/src/')

from gustav import ebi, ncbi, nlm, biogrid, nih, openalex
from gustav import publications
from gustav import github
from gustav import access_framework
from gustav import mapper

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import auc
from scipy.stats import fisher_exact
pd.options.display.precision = 3
pd.options.display.expand_frame_repr = False
pd.options.display.max_columns = 20

In [4]:
rw_db = pd.read_csv('../data/240903_retraction_watch_db.csv', encoding='latin')
rw_db = rw_db[(rw_db['RetractionNature'] == 'Retraction')]
rw_db['OriginalPaperDOI'] = rw_db['OriginalPaperDOI'].astype(str).str.lower()
rw_db['doi'] = rw_db['OriginalPaperDOI'].astype(str).apply(lambda x: 'https://doi.org/' + x).values

In [3]:
label_df = pd.read_csv('../data/240107_combined_sem_labels.csv')

meta_df = pd.read_csv('../data/240107_combined_sem_metadata.csv')

works_df = pd.read_csv('../data/240107_combined_sem_works.csv')

meta_df['brandmarks'] = meta_df['doi'].isin(label_df['doi'])
meta_df['problematic'] = meta_df['doi'].isin(label_df[label_df['problematic']]['doi'])

works_df['sem'] = works_df['doi'].isin(meta_df['doi'])
works_df['brandmarks'] = works_df['doi'].isin(label_df['doi'])
works_df['problematic'] = works_df['doi'].isin(label_df[label_df['problematic']]['doi'])

meta_df['brandmarks'] = meta_df['doi'].isin(label_df['doi'])

meta_df['problematic'] = meta_df['doi'].isin(label_df[label_df['problematic']]['doi'])

works_df['sem'] = works_df['doi'].isin(meta_df['doi'])

works_df['brandmarks'] = works_df['doi'].isin(label_df['doi'])

works_df['problematic'] = works_df['doi'].isin(label_df[label_df['problematic']]['doi'])

works_df['no_id'] = works_df['doi'].isin(label_df[~label_df['id_instrument']]['doi'])

works_df['exists'] = True

works_df['label'] = ''
works_df.loc[~works_df['sem'], 'label'] = 'no_sem'
works_df.loc[works_df['sem'] & ~works_df['brandmarks'], 'label'] = 'sem_no_brandmarks'
works_df.loc[works_df['brandmarks'] & ~works_df['problematic'] & ~works_df['no_id'], 'label'] = 'brandmarks_id_correct'
works_df.loc[works_df['brandmarks'] & works_df['no_id'], 'label'] = 'brandmarks_no_id'
works_df.loc[works_df['problematic'], 'label'] = 'problematic'

C:\Users\richa\AppData\Local\Temp\ipykernel_34264\1777301440.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_df = pd.read_csv('../data/240107_combined_sem_metadata.csv')
C:\Users\richa\AppData\Local\Temp\ipykernel_34264\1777301440.py:5: DtypeWarning: Columns (8,11,20) have mixed types. Specify dtype option on import or set low_memory=False.
  works_df = pd.read_csv('../data/240107_combined_sem_works.csv')


In [11]:
works_df[works_df['doi'].apply(lambda x: 'https://doi.org/' + x).isin(rw_db['doi'])].sort_values('problematic')[['doi', 'problematic']]

,doi,problematic
1074,10.1016/j.ceramint.2012.02.021,False
870048,10.1371/journal.pone.0167868,False
869779,10.1371/journal.pone.0173434,False
869640,10.1371/journal.pone.0177625,False
869117,10.1371/journal.pone.0180487,False
...,...,...
606926,10.1038/s41598-022-16666-6,False
605151,10.1038/s41598-022-06049-2,False
605112,10.1038/s41598-022-07005-w,False
616409,10.1038/s41598-022-13692-2,False


In [8]:
rw_db['doi']

0            https://doi.org/10.1155/2022/2557696
1            https://doi.org/10.1155/2022/3873484
2            https://doi.org/10.1155/2021/5580761
3            https://doi.org/10.1155/2021/2223344
4            https://doi.org/10.1155/2022/2342312
                           ...                   
55806    https://doi.org/10.1001/jama.298.13.1539
55807         https://doi.org/10.1128/jb.00242-13
55808         https://doi.org/10.1128/jb.01375-10
55809           https://doi.org/10.1002/jcb.21852
55810           https://doi.org/10.1002/jcb.23190
Name: doi, Length: 51022, dtype: object